# Challenge Locaweb EDA

## Configurações Iniciais

### Configurações a depender do ambiente

In [2]:
import os
import sys
import subprocess

# --- CONFIGURAÇÃO ALVO ---
TARGET_PYSPARK = "4.1.1"

# 1. Identifica o Ambiente (Fora da função para ser global)
IN_COLAB = 'google.colab' in sys.modules
ENV_NAME = "☁️ Google Colab" if IN_COLAB else "💻 Ambiente Local (WSL/Jupyter)"

print(f"Detectado: {ENV_NAME}")
print(f"Versão do Python: {sys.version.split()[0]}")

# 2. Define os Caminhos Globais
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = "/content/drive/MyDrive/fiap/segundo ano/challenge_locaweb/"
else:
    BASE_PATH = "raw data/" # Adicionei a barra no final para facilitar o join

def setup_pyspark():
    if IN_COLAB:
        try:
            import pyspark
            if pyspark.__version__ != TARGET_PYSPARK:
                print(f"(!) Atualizando PySpark de {pyspark.__version__} para {TARGET_PYSPARK}...")
                subprocess.check_call([sys.executable, "-m", "pip", "install", f"pyspark=={TARGET_PYSPARK}", "-q"])
                print("🚨 Reinicie o Ambiente (Runtime > Restart Session) para aplicar a mudança!")
        except ImportError:
            print(f"(!) Instalando PySpark {TARGET_PYSPARK}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", f"pyspark=={TARGET_PYSPARK}", "-q"])

    # Verificação Final
    try:
        import pyspark
        if pyspark.__version__ == TARGET_PYSPARK:
            print(f"✅ PySpark {pyspark.__version__} pronto!")
        else:
            print(f"⚠️ Alerta: PySpark está na versão {pyspark.__version__}. Alvo era {TARGET_PYSPARK}.")
    except ImportError:
        print("❌ Erro: PySpark não encontrado.")

setup_pyspark()

Detectado: 💻 Ambiente Local (WSL/Jupyter)
Versão do Python: 3.12.3
✅ PySpark 4.1.1 pronto!


### Importação de Bibliotecas

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import * # Para definir Schemas (StructType, DoubleType, etc)
from pyspark.sql.window import Window

### Inicialização da Sessão Spark

In [4]:
spark = SparkSession.builder \
    .appName("DataExploration") \
    .master("local[*]") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/28 21:35:56 WARN Utils: Your hostname, PCJULIA, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/28 21:35:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/28 21:35:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
spark

### Importação de Dados

In [25]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ";") \
    .option("encoding", "ISO-8859-1") \
    .load(f"{BASE_PATH}/LW-DATASET-CSV.CSV")

In [26]:
df.show(20)

+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+---------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+---------------+----------------+------------+
|    Número|Prioridade|Produto|Categoria|Subcategoria|Grupo designado|Item de configuração|             Aberto|Resolvido|          Encerrado|Duração|Código de fechamento|  Descrição resumida|Solução|   Aberto por|Incidente Pai|         Status|Entrou para KPI?|KPI Violado?|
+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+---------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+---------------+----------------+------------+
|INC8654273| 3 - Média|   NULL|     NULL|        NULL|         Team14|             IC00001|2025-12-31 23:45:18|     NULL|2025-12-31 23:45:32|     14|                NULL|Problem:

In [27]:
df.printSchema()

root
 |-- Número: string (nullable = true)
 |-- Prioridade: string (nullable = true)
 |-- Produto: string (nullable = true)
 |-- Categoria: string (nullable = true)
 |-- Subcategoria: string (nullable = true)
 |-- Grupo designado: string (nullable = true)
 |-- Item de configuração: string (nullable = true)
 |-- Aberto: timestamp (nullable = true)
 |-- Resolvido: timestamp (nullable = true)
 |-- Encerrado: timestamp (nullable = true)
 |-- Duração: integer (nullable = true)
 |-- Código de fechamento: string (nullable = true)
 |-- Descrição resumida: string (nullable = true)
 |-- Solução: string (nullable = true)
 |-- Aberto por: string (nullable = true)
 |-- Incidente Pai: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Entrou para KPI?: string (nullable = true)
 |-- KPI Violado?: string (nullable = true)



## Validação da Base

In [28]:
colunas_originais = df.columns

In [29]:
def identificar_duplicatas(df, nome_coluna):
    """
    Adiciona uma coluna booleana indicando se o valor na coluna informada é duplicado.
    """
    window_spec = Window.partitionBy(nome_coluna)
    return df.withColumn(
        f"{nome_coluna}_is_duplicate", 
        F.count(nome_coluna).over(window_spec) > 1
    )

In [30]:
def identificar_nulos(df, nome_coluna):
    """
    Adiciona uma coluna booleana indicando se o valor na coluna informada é nulo.
    """
    return df.withColumn(
        f"{nome_coluna}_is_null", 
        F.col(nome_coluna).isNull()
    )

In [31]:
# Criação de Coluna "Deve ser retirado da base"
df = df.withColumn(
    "Deve_ser_retirado_da_base",
    F.lit(False)  # Inicialmente, todos são False
)

In [32]:
# Verificação de Unicidade da Chave Primária
total_registros = df.count()
registros_distintos = df.select("Número").distinct().count()
unicidade_chave = total_registros == registros_distintos
if unicidade_chave:
    print("✅ A coluna 'Número' é única.")
else:
    print("⚠️ A coluna 'Número' contém duplicatas.")
    print(f"Número de duplicatas: {total_registros - registros_distintos} ({(total_registros - registros_distintos)/total_registros*100:.2f}%)")
    df = identificar_duplicatas(df, "Número")
    df.filter(F.col("Número_is_duplicate") == True).orderBy("Número").show(20)


⚠️ A coluna 'Número' contém duplicatas.
Número de duplicatas: 8 (0.01%)
+----------+----------+-------------+----------+---------------+---------------+--------------------+------+---------+---------+-------+--------------------+------------------+-------+----------+-------------+------+----------------+------------+-------------------------+-------------------+
|    Número|Prioridade|      Produto| Categoria|   Subcategoria|Grupo designado|Item de configuração|Aberto|Resolvido|Encerrado|Duração|Código de fechamento|Descrição resumida|Solução|Aberto por|Incidente Pai|Status|Entrou para KPI?|KPI Violado?|Deve_ser_retirado_da_base|Número_is_duplicate|
+----------+----------+-------------+----------+---------------+---------------+--------------------+------+---------+---------+-------+--------------------+------------------+-------+----------+-------------+------+----------------+------------+-------------------------+-------------------+
|Did you m"|      NULL|Monitoramento|      NULL|S

Os dados com chaves duplicadas causam ruido, eles representam uma parcela mínima da base e possuem muitas colunas nulas, por isso devem ser retirados da base tratada.

In [33]:
# Muda a flag "Deve ser retirado da_base" para True para os registros duplicados
df = df.withColumn(
    "Deve_ser_retirado_da_base",
    F.when(F.col("Número_is_duplicate") == True, True).otherwise(F.col("Deve_ser_retirado_da_base"))
)

In [34]:
# Validar valores nulos em colunas que não deveriam ter
colunas_nao_nulas = [
    "Número",
    "Prioridade",
    "Grupo designado",
    "Aberto",
    "Encerrado",
    "Duração",
    "Descrição Resumida",
    "Aberto por",
    "Status",
    "Entrou para KPI?",
    "KPI Violado?"
]
nulos_por_coluna = []
for coluna in colunas_nao_nulas:
    df = identificar_nulos(df, coluna)
    nulos_count = df.filter(F.col(f"{coluna}_is_null") == True).count()
    nulos_por_coluna.append(nulos_count)
    if nulos_count > 0:
        print(f"⚠️ A coluna '{coluna}' contém {nulos_count} valores nulos - ({nulos_count/df.count()*100:.2f}%).")
        print(f"Dentre estes, {df.filter(F.col(f'{coluna}_is_null') == True).filter(F.col('Deve_ser_retirado_da_base') == False).count()} estão em registros que não foram descartados anteriormente.")
        df.filter(F.col(f"{coluna}_is_null") == True).filter(F.col('Deve_ser_retirado_da_base') == False).select(["Número", coluna, f"{coluna}_is_null"]).show(20)
    else:
        print(f"✅ A coluna '{coluna}' não contém valores nulos.")

validacao_nulos = {
    "Coluna": colunas_nao_nulas,
    "Nulos": nulos_por_coluna,
    "Percentual Nulos (%)": [f"{(nulos/df.count()*100):.2f}%" for nulos in nulos_por_coluna]
}

✅ A coluna 'Número' não contém valores nulos.
⚠️ A coluna 'Prioridade' contém 11 valores nulos - (0.01%).
Dentre estes, 2 estão em registros que não foram descartados anteriormente.
+--------------------+----------+------------------+
|              Número|Prioridade|Prioridade_is_null|
+--------------------+----------+------------------+
| Is the server ru...|      NULL|              true|
| Is the server ru...|      NULL|              true|
+--------------------+----------+------------------+

✅ A coluna 'Grupo designado' não contém valores nulos.
⚠️ A coluna 'Aberto' contém 11 valores nulos - (0.01%).
Dentre estes, 2 estão em registros que não foram descartados anteriormente.
+--------------------+------+--------------+
|              Número|Aberto|Aberto_is_null|
+--------------------+------+--------------+
| Is the server ru...|  NULL|          true|
| Is the server ru...|  NULL|          true|
+--------------------+------+--------------+

⚠️ A coluna 'Encerrado' contém 11 valores

In [ ]:
# Validação de lista de valores permitidos por coluna
colunas_com_valores_definidos = {
    "Prioridade": ["1 - Crítica","2 - Alta","3 - Média","4 - Baixa","5 - Muito Baixa"],
    "Solução": ["Contorno", "Definitiva", ""],
    "Aberto por": ["Manual", "Monitoramento"],
    "Status": ["Aguardando Problema", "Encerrado", "Encerrado Automaticamente", "Sem Intervenção"],
    "Entrou para KPI?": ["SIM", "NAO"],
    "KPI Violado?": ["SIM", "NAO"]
}